# 01 — Member turns, segmented into numbered sentences

**Purpose.** Prepare the Track-2 extraction input: every legislator turn in the
SST sample, split into numbered sentences by a pinned deterministic segmenter.
The stance records that Stage 3 extracts point at these sentence numbers, so
this table is the ground every downstream evidence pointer stands on.

**Who counts as a legislator.** A spoken turn whose speaker is on the committee
roster (`is_member`) **or** is the presiding officer — Senate transcripts record
the chair as `The Chairman` with no surname, carried in the data as the
`bioGuideId == '0000000'` sentinel with `is_member == False`. Excluding the
chair would drop the hearing's most frequent speaker, so the rule is
`is_member OR chair`. Witnesses and unattributed front matter are excluded:
Track 2 measures member positions (the Stage-3 plan's locked estimand), and
witness content reaches the pipeline through Track 1.

**Why segmentation is deterministic and version-pinned.** The model selects
sentence *numbers*, never text, which is what makes evidence hallucination
structurally impossible — but only if the numbering is reproducible. `stance.py`
refuses to import against any pysbd version other than the one recorded there.

**Inputs.** `data/raw/02_hearings/hearings_sst.parquet` — 2,352 hearings.

**Outputs.** `interim/topics/track2/member_turns.parquet` on scratch — one row
per legislator turn: identifiers, speaker fields, and the sentence list.

**Gate (per the Stage-3 plan §8).** Turn and sentence counts measured (the
plan's 200–250k figure was an estimate; this notebook replaces it with a
number), segmenter pinned, 50-turn segmentation spot-check printed for eyeball
audit.

## Setup

Paths come from `lib/paths.py` and the segmenter from `lib/stance.py`, which
asserts its pysbd version on import — a wrong environment fails here, on the
first cell, not after an hour of segmentation.

In [1]:
import sys
import time
from concurrent.futures import ProcessPoolExecutor

sys.path.insert(0, "../lib")

import numpy as np
import pandas as pd

import stance
from paths import HEARINGS_REPO, topics_run_dir

RUN_DIR = topics_run_dir("track2")
TARGET = RUN_DIR / "member_turns.parquet"
CHAIR_ID = "0000000"

print(f"segmenter : pysbd {stance.PYSBD_VERSION} (asserted on import)")
print(f"output    : {TARGET}")

segmenter : pysbd 0.3.4 (asserted on import)
output    : /scratch2/acroh/projects/llm_method/data/interim/topics/track2/member_turns.parquet


## Step 1 — Select legislator turns

`turn_idx` is the turn's position in the hearing's full turn list, kept so any
record can be traced back to the exact place in the source table — including
past the witness turns this step drops. The composition table printed here is
the first measured answer to "how much of a hearing is member speech", which
the convergence analysis later conditions on (witness-heavy hearings are
*expected* to diverge between tracks).

In [2]:
sst = pd.read_parquet(HEARINGS_REPO / "hearings_sst.parquet",
                      columns=["packageId", "year", "chamber", "turns"])

rows = []
composition = {"member": 0, "chair": 0, "witness_or_other": 0, "not_spoken": 0}
for hearing in sst.itertuples():
    for turn_idx, turn in enumerate(hearing.turns):
        if turn.get("section") != "spoken":
            composition["not_spoken"] += 1
            continue
        is_chair = turn.get("bioGuideId") == CHAIR_ID
        if not (turn.get("is_member") or is_chair):
            composition["witness_or_other"] += 1
            continue
        composition["chair" if is_chair else "member"] += 1
        rows.append({
            "packageId": hearing.packageId,
            "year": int(hearing.year),
            "chamber": hearing.chamber,
            "turn_idx": turn_idx,
            "bioGuideId": turn.get("bioGuideId"),
            "surname": turn.get("surname"),
            "honorific": turn.get("honorific"),
            "is_chair": is_chair,
            "n_words": int(turn.get("n_words") or 0),
            "text": turn["text"],
        })

turns = pd.DataFrame(rows)
total_spoken = sum(v for k, v in composition.items() if k != "not_spoken")
print(f"hearings          : {len(sst):,}")
for kind, count in composition.items():
    print(f"  {kind:<17}: {count:>8,}")
print(f"legislator turns  : {len(turns):,} "
      f"({len(turns) / total_spoken:.0%} of spoken turns)")
print(f"legislator words  : {turns['n_words'].sum() / 1e6:.1f}M")
print(f"hearings w/ none  : {len(sst) - turns['packageId'].nunique():,}")

hearings          : 2,352
  member           :  197,985
  chair            :   23,326
  witness_or_other :  177,718
  not_spoken       :       12
legislator turns  : 221,311 (55% of spoken turns)
legislator words  : 38.6M
hearings w/ none  : 9


## Step 2 — Segment, then clean

Parallel over row chunks because pysbd is pure-Python and CPU-bound; the
result is order-preserving, so parallelism cannot change the indices.

Segmentation is followed by `stance.clean_turn_sentences`, four deterministic
rules whose need was **measured, not assumed** (first prep run + first dev
run, 2026-08-22):

1. **Truncate at adjournment / inserted-document markers.** The
   post-adjournment appendix rides on the last speaker's turn — 748,830
   sentences, 35.4% of the raw corpus — and `[The prepared statement of X
   follows:]` blocks sit inside mid-hearing turns; both are written material
   that would be extracted as member speech, and the inserted documents
   caused every JSON failure in the first extraction dev run.
2. **Drop the leading speaker-label echo** ("Senator Dorgan.") — 99% of turns
   open with it.
3. **Drop bracketed stage directions** ("[Laughter.]").
4. **Drop all-caps inline headers** ("STATEMENT OF MR. ...").

Every rule's removal count is printed below — a filter whose removals are not
reported is a silent sample choice.

In [3]:
def _segment_batch(texts):
    # segment_turn works paragraph block by block — the guard against
    # missegmented mega-"sentences" (see its docstring; measured fix:
    # 2,421-word blob -> max 64 words).
    return [stance.segment_turn(t) for t in texts]

# Worker count follows the SLURM allocation, not the node's core count —
# os.cpu_count() on a shared node reports cores we do not own.
import os
from collections import Counter
n_workers = int(os.environ.get("SLURM_CPUS_PER_TASK", os.cpu_count() or 4))

started = time.time()
texts = turns["text"].tolist()
batches = [texts[i:i + 2000] for i in range(0, len(texts), 2000)]
with ProcessPoolExecutor(max_workers=n_workers) as pool:
    segmented = [s for batch in pool.map(_segment_batch, batches) for s in batch]

raw_total = sum(len(s) for s in segmented)
print(f"segmented in {time.time() - started:.0f}s on {n_workers} workers "
      f"({raw_total:,} raw sentences)")

# Deterministic cleanup (see the markdown above); per-rule counts reported.
cleaned, removal_totals = [], Counter()
for sentences, surname, honorific in zip(segmented, turns["surname"],
                                          turns["honorific"]):
    kept, removed = stance.clean_turn_sentences(sentences, surname, honorific)
    cleaned.append(kept)
    removal_totals.update(removed)

turns["sentences"] = cleaned
turns["n_sentences"] = turns["sentences"].str.len()

print("removed by rule:")
for rule, count in removal_totals.most_common():
    print(f"  {rule:<16}: {count:>9,} ({count / raw_total:.1%} of raw)")

n_sent = turns["n_sentences"]
print(f"\nsentences kept    : {n_sent.sum():,} "
      f"({n_sent.sum() / raw_total:.0%} of raw)")
print(f"per turn          : median {n_sent.median():.0f}, "
      f"p90 {n_sent.quantile(0.9):.0f}, max {n_sent.max():,}")
print(f"emptied by cleanup: {(n_sent == 0).sum():,} turns "
      f"(procedural-only; extraction skips them)")

# Residual missegmentation check: a >150-word "sentence" inside ONE paragraph
# is a punctuation-free blob the paragraph guard cannot help; count them.
long_left = sum(1 for ss in turns["sentences"] for s in ss
                if len(s.split()) > 150)
print(f"sentences >150 words after paragraph-wise segmentation: {long_left:,}")
short = turns["sentences"].map(
    lambda ss: sum(len(s) < stance.MIN_SENTENCE_CHARS for s in ss)).sum()
print(f"short fragments (<{stance.MIN_SENTENCE_CHARS} chars): {short:,} "
      f"({short / max(n_sent.sum(), 1):.1%} of kept)")

segmented in 42s on 32 workers (2,490,103 raw sentences)


removed by rule:
  truncated       : 1,249,776 (50.2% of raw)
  label_echo      :   219,650 (8.8% of raw)
  header          :    28,666 (1.2% of raw)
  stage_direction :     6,350 (0.3% of raw)

sentences kept    : 985,661 (40% of raw)
per turn          : median 2, p90 9, max 3,831
emptied by cleanup: 1,151 turns (procedural-only; extraction skips them)


sentences >150 words after paragraph-wise segmentation: 237
short fragments (<15 chars): 105,310 (10.7% of kept)


## Step 3 — Spot-check 50 segmentations

The gate requires an eyeball audit: hearing speech is full of abbreviations
(`Mr.`, `H.R. 3684`, `No. 5`, `U.S.`) that a naive splitter breaks on. The
sample is seeded so re-running the notebook re-prints the same 50 turns.

In [4]:
rng = np.random.default_rng(20260820)
sample = turns.loc[turns["n_sentences"] >= 3].sample(50, random_state=1089)

for row in sample.head(8).itertuples():
    who = f"{row.honorific or ''} {row.surname or ''}".strip()
    print("=" * 78)
    print(f"{row.packageId}  turn {row.turn_idx}  [{who}]")
    print(stance.format_numbered(row.sentences[:8]))
    if len(row.sentences) > 8:
        print(f"  ... {len(row.sentences) - 8} more sentences")
print("=" * 78)
print(f"(8 of the 50 sampled turns shown; the parquet holds all — inspect",
      "further in Jupyter as needed)")

CHRG-111shrg66169  turn 84  [Senator Lautenberg]
[1] Thank you very much.
[2] And now like to hear from Joseph Kelly, who's the Acting Chief of New Jersey Transit Police.
[3] Mr. Kelly, welcome.
[4] NJ TRANSIT
CHRG-115hhrg26233  turn 31  [Mr. Bera]
[1] Okay.
[2] And then also, obviously, these are multiyear missions, multiyear strategic planning as we're looking at longer-term goals.
[3] You know, the Chairman brought up the International Space Station and we've committed to funding through 2024.
[4] Obviously, that's one thing we've certainly been meeting with folks, chatting with folks at NASA, as well as others, the potential possibility as more commercial entities get into space, as other institutions see this valuable asset and the academic sector and others, the potential of life after 2024.
[5] And I'd be curious in NASA's long-term planning how are you guys thinking about the ISS in longer-term?
CHRG-107shrg89680  turn 4  [Senator Burns]
[1] Senator Burns [continuing].
[2] So t

## Step 4 — Persist

The raw `text` column is dropped: it survives in `hearings_sst.parquet` and
keeping one copy of every long string is the difference between a working
table and a second corpus. `(packageId, turn_idx)` is the join key back.

In [5]:
out = turns.drop(columns=["text"]).reset_index(drop=True)
out.to_parquet(TARGET, compression="zstd", index=False)
print(f"wrote {TARGET}")
print(f"  {len(out):,} rows, {TARGET.stat().st_size / 1e6:.1f} MB")

wrote /scratch2/acroh/projects/llm_method/data/interim/topics/track2/member_turns.parquet
  221,311 rows, 38.4 MB


## Verification

Checked on the file as re-read from disk, not on the in-memory frame.

In [6]:
back = pd.read_parquet(TARGET)

assert not back.duplicated(["packageId", "turn_idx"]).any(), "duplicate turn key"
print(f"1) key (packageId, turn_idx) unique across {len(back):,} rows")

assert (back["sentences"].str.len() == back["n_sentences"]).all()
empty = int((back["n_sentences"] == 0).sum())
print(f"2) sentence counts consistent; {empty} empty turns "
      f"({empty / len(back):.2%}) — tolerated, extraction skips them")

assert back["is_chair"].any(), "no chair turns — the selection rule regressed"
print(f"3) chair turns present: {int(back['is_chair'].sum()):,} "
      f"({back['is_chair'].mean():.0%} of legislator turns)")

assert back["packageId"].nunique() >= 0.95 * len(sst), (
    "more than 5% of hearings contribute no legislator turns")
print(f"4) {back['packageId'].nunique():,} of {len(sst):,} hearings contribute turns")

print(f"5) segmenter pinned: pysbd {stance.PYSBD_VERSION}")
print("\nDone. Next: 02_extraction_promptbook.ipynb (dev sample, prompt iteration).")

1) key (packageId, turn_idx) unique across 221,311 rows
2) sentence counts consistent; 1151 empty turns (0.52%) — tolerated, extraction skips them
3) chair turns present: 23,326 (11% of legislator turns)
4) 2,343 of 2,352 hearings contribute turns
5) segmenter pinned: pysbd 0.3.4

Done. Next: 02_extraction_promptbook.ipynb (dev sample, prompt iteration).
